In [22]:
import numpy as np
import cv2
import time
import os
import glob
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

from motion_detection.utils.motion_utils import *

In [2]:
def simulate_low_fps(sequence_data, target_frames):
    """
    mô phỏng giảm fps về target frames, ở đây ta nhận vô video gốc 
    """
    total_frames = len(sequence_data)
    
    if total_frames <= target_frames:
        return np.copy(sequence_data)
    
    # Lấy các index rải đều từ đầu đến cuối video
    indices = np.linspace(0, total_frames - 1, target_frames).astype(int)
    
    return sequence_data[indices]

def simulate_missing_tracking(sequence_data, missing_ratio=0.2, missing_val=-1.0):
    """
    Cố tình tạo missing keypoint frame trong dữ liệu bằng cách gán -1.0 cho một số frame ngẫu nhiên.
    Mô phỏng tình huống bị vật cản che tay hoặc chuyển động quá nhanh làm mất tracking.
    """
    seq_dropped = np.copy(sequence_data)
    num_frames = len(seq_dropped)
    
    num_drop = int(num_frames * missing_ratio)
    
    # Tránh drop frame đầu tiên và cuối cùng để hàm nội suy tuyến tính hoạt động đẹp nhất
    if num_frames > 2:
        drop_indices = np.random.choice(range(1, num_frames - 1), size=num_drop, replace=False)
    else:
        drop_indices = []
        
    # Giả lập mất tracking tay trái (0-62) hoặc tay phải (63-125) ngẫu nhiên
    for idx in drop_indices:
        # Random: 0 là mất tay trái, 1 là mất tay phải, 2 là mất cả hai tay
        lose_type = np.random.choice([0, 1, 2])
        if lose_type == 0 or lose_type == 2:
            seq_dropped[idx, 0:63] = missing_val
        if lose_type == 1 or lose_type == 2:
            seq_dropped[idx, 63:126] = missing_val
            
    return seq_dropped

In [12]:
# collect visualize video
SAVE_DIR = "data/visualize_raw"
VID_TYPE = ["clapping", "shaking"]
DURATION =  2.0
BREAK_DURATION = 3

os.makedirs(SAVE_DIR, exist_ok=True)

cap = cv2.VideoCapture(1)

# camera resolution
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
print(width, height)
print("Press 'q' to exit!")

for type in VID_TYPE:
    vid_path = os.path.join(SAVE_DIR, f"{type}.mp4")
    out = cv2.VideoWriter(vid_path, fourcc, 60, frameSize=(width, height))

    start_break = time.time()
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break

        elapsed_break = time.time() - start_break
        remaining_break = int(BREAK_DURATION - elapsed_break)

        if remaining_break == 0:
            break

        cv2.putText(frame, f"GET READY: {int(remaining_break) + 1}s", (50, 80), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 255), 3)

        cv2.putText(frame, f"Next: Video {type}", (50, 140), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 0), 2)

        cv2.imshow("Data Collector", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    print(f"Start recording {type} video!")
    start_record = time.time()
    frames_collected = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break

        elapsed_record = time.time() - start_record
        if elapsed_record >  DURATION:
            break

        out.write(frame)
        frames_collected += 1

        display_frame = frame.copy()

        cv2.putText(display_frame, f"RECORDING VIDEO {type}", (50, 80), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 255), 3)
        
        cv2.rectangle(display_frame, (50, 100), (50 + int((elapsed_record/DURATION)*400), 120), (0, 0, 255), -1)
        
        cv2.imshow("Data Collector", display_frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    out.release()
    print(f"-> Saved: {vid_path} | Collected: {frames_collected} frames")

cap.release()
cv2.destroyAllWindows()
print("Done!")

1280 720
Press 'q' to exit!
Start recording clapping video!
-> Saved: data/visualize_raw\clapping.mp4 | Collected: 125 frames
Start recording shaking video!
-> Saved: data/visualize_raw\shaking.mp4 | Collected: 125 frames
Done!


In [32]:
def extract_video_to_sequence(video_path, model_path='motion_detection/models/hand_landmarker.task'):
    """
    Đọc video, trả về mảng numpy shape (Số frame, 126) sử dụng MediaPipe Tasks API.
    """
    # 1. Cấu hình HandLandmarker
    base_options = python.BaseOptions(model_asset_path=model_path)
    options = vision.HandLandmarkerOptions(
        base_options=base_options,
        running_mode=vision.RunningMode.VIDEO, # Chế độ tối ưu cho video (có tracking)
        num_hands=2,
        min_hand_detection_confidence=0.5,
        min_hand_presence_confidence=0.5,
        min_tracking_confidence=0.5)
        
    sequence = []
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps == 0: fps = 30.0 # Fallback nếu không đọc được fps
    
    frame_idx = 0
    
    with vision.HandLandmarker.create_from_options(options) as landmarker:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
            
            image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)
            
            timestamp_ms = int((frame_idx / fps) * 1000)
            
            result = landmarker.detect_for_video(mp_image, timestamp_ms)
            
            frame_data = np.full(126, -1.0)
            
            if result.hand_landmarks:
                # result.hand_landmarks là list chứa tọa độ các tay
                # result.handedness là list chứa thông tin Trái/Phải tương ứng
                for idx, hand_landmarks in enumerate(result.hand_landmarks):
                    hand_label = result.handedness[idx][0].category_name
                    
                    coords = []
                    for lm in hand_landmarks:
                        coords.extend([lm.x, lm.y, lm.z])
                        
                    if hand_label == 'Left':
                        frame_data[0:63] = coords
                    else:
                        frame_data[63:126] = coords
                        
            sequence.append(frame_data)
            frame_idx += 1
            
    cap.release()
    return np.array(sequence)


RAW_DIR = "data/visualize_raw"
OUT_DIR = "data/visualize_npy"
VID_TYPE = ["clapping", "shaking"]

os.makedirs(OUT_DIR, exist_ok=True)

for v_type in VID_TYPE:
    video_path = os.path.join(RAW_DIR, f"{v_type}.mp4")
    print(f"\nProcessing video: {video_path} ...")
    
    raw_sequence = extract_video_to_sequence(video_path)
    print(f" -> Extracted: {raw_sequence.shape} frames")
    
    # 2. Tạo version 30 frames 
    seq_30_fps = simulate_low_fps(raw_sequence, target_frames=30)
    seq_30_fps_missing = simulate_missing_tracking(seq_30_fps, missing_ratio=0.3)
    
    # 3. Tạo version 40 frames 
    seq_40_fps = simulate_low_fps(raw_sequence, target_frames=40)
    seq_40_fps_missing = simulate_missing_tracking(seq_40_fps, missing_ratio=0.3)
    
    # 4. Lưu lại thành file .npy 
    path_30 = os.path.join(OUT_DIR, f"{v_type}_30fps_missing.npy")
    path_40 = os.path.join(OUT_DIR, f"{v_type}_40fps_missing.npy")
    
    np.save(path_30, seq_30_fps_missing)
    np.save(path_40, seq_40_fps_missing)
    
    print(f" -> Saved: {path_30} (Shape: {seq_30_fps.shape})")
    print(f" -> Saved: {path_40} (Shape: {seq_40_fps_missing.shape})")

print("\nDone!")


Processing video: data/visualize_raw\clapping.mp4 ...
 -> Extracted: (125, 126) frames
 -> Saved: data/visualize_npy\clapping_30fps_missing.npy (Shape: (30, 126))
 -> Saved: data/visualize_npy\clapping_40fps_missing.npy (Shape: (40, 126))

Processing video: data/visualize_raw\shaking.mp4 ...
 -> Extracted: (125, 126) frames
 -> Saved: data/visualize_npy\shaking_30fps_missing.npy (Shape: (30, 126))
 -> Saved: data/visualize_npy\shaking_40fps_missing.npy (Shape: (40, 126))

Done!


In [33]:
INPUT_DIR = "data/visualize_npy/"
OUTPUT_DIR = "data/visualize_processed"

os.makedirs(OUTPUT_DIR, exist_ok=True)

npy_files = glob.glob(os.path.join(INPUT_DIR, "*.npy"))

print(f"found {len(npy_files)} raw data")

for file_path in npy_files:
    filename = os.path.basename(file_path)
    print(f"\nProcessing: {filename}")
    
    raw_data = np.load(file_path)
    print(f" -> Input Shape: {raw_data.shape}")
    
    processed_data = visualizer_pipeline(raw_data)
    
    out_path = os.path.join(OUTPUT_DIR, f"processed_{filename}")
    np.save(out_path, processed_data)
    
    print(f" -> Output Shape: {processed_data.shape}")
    print(f" -> Saved: {out_path}")

print("\nDone!")

found 4 raw data

Processing: clapping_30fps_missing.npy
 -> Input Shape: (30, 126)
 -> Output Shape: (50, 126)
 -> Saved: data/visualize_processed\processed_clapping_30fps_missing.npy

Processing: clapping_40fps_missing.npy
 -> Input Shape: (40, 126)
 -> Output Shape: (50, 126)
 -> Saved: data/visualize_processed\processed_clapping_40fps_missing.npy

Processing: shaking_30fps_missing.npy
 -> Input Shape: (30, 126)
 -> Output Shape: (50, 126)
 -> Saved: data/visualize_processed\processed_shaking_30fps_missing.npy

Processing: shaking_40fps_missing.npy
 -> Input Shape: (40, 126)
 -> Output Shape: (50, 126)
 -> Saved: data/visualize_processed\processed_shaking_40fps_missing.npy

Done!


In [34]:
HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),           # Ngón cái
    (0, 5), (5, 6), (6, 7), (7, 8),           # Ngón trỏ
    (5, 9), (9, 10), (10, 11), (11, 12),      # Ngón giữa
    (9, 13), (13, 14), (14, 15), (15, 16),    # Ngón áp út
    (13, 17), (17, 18), (18, 19), (19, 20),   # Ngón út
    (0, 17)                                   # Nối cổ tay với ngón út
]

def draw_hand_on_canvas(canvas, hand_pts, color_dot, color_line):
    """
    hand_pts: mảng (21, 3) chứa tọa độ x, y, z của 1 tay
    canvas: bức ảnh OpenCV (numpy array)
    """
    height, width, _ = canvas.shape
    
    # Do dữ liệu đã được chuẩn hóa quanh tâm 0 (scale max ~1.0)
    # Ta cần phóng to nó lên và dời tâm ra giữa canvas
    scale = 300 
    offset_x, offset_y = width // 2, height // 2
    
    # 1. Chuyển đổi tọa độ sang pixel
    pixel_pts = []
    for pt in hand_pts:
        # Nếu gặp điểm -1.0 (tay tàng hình), lưu None để không vẽ
        if pt[0] == -1.0:
            pixel_pts.append(None)
        else:
            px = int(pt[0] * scale + offset_x)
            py = int(pt[1] * scale + offset_y)
            pixel_pts.append((px, py))
            
    # 2. Vẽ các đường nối (Bones)
    for connection in HAND_CONNECTIONS:
        idx1, idx2 = connection
        pt1 = pixel_pts[idx1]
        pt2 = pixel_pts[idx2]
        
        if pt1 is not None and pt2 is not None:
            cv2.line(canvas, pt1, pt2, color_line, 2)
            
    # 3. Vẽ các điểm khớp (Joints)
    for pt in pixel_pts:
        if pt is not None:
            cv2.circle(canvas, pt, 4, color_dot, -1)

INPUT_DIRS = ["data/visualize_npy", "data/visualize_processed"]
OUTPUT_DIR = "data/visualize"
os.makedirs(OUTPUT_DIR, exist_ok=True)

npy_files = []
for d in INPUT_DIRS:
    npy_files.extend(glob.glob(os.path.join(d, "*.npy")))
    
print(f"Tìm thấy {len(npy_files)} file .npy.")

TARGET_DURATION = 4.0 
CANVAS_SIZE = 800 

for file_path in npy_files:
    filename = os.path.basename(file_path)
    vid_name = filename.replace(".npy", ".mp4")
    out_path = os.path.join(OUTPUT_DIR, vid_name)
    
    seq_data = np.load(file_path)
    num_frames = len(seq_data)
    
    # 4s video
    FPS = num_frames / TARGET_DURATION
    
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(out_path, fourcc, FPS, (CANVAS_SIZE, CANVAS_SIZE))
    
    for frame_idx in range(num_frames):
        canvas = np.zeros((CANVAS_SIZE, CANVAS_SIZE, 3), dtype=np.uint8)
        
        frame_coords = seq_data[frame_idx]
        
        left_hand = frame_coords[0:63].reshape(21, 3)
        right_hand = frame_coords[63:126].reshape(21, 3)
        
        draw_hand_on_canvas(canvas, left_hand, color_dot=(255, 150, 0), color_line=(255, 50, 0))
        draw_hand_on_canvas(canvas, right_hand, color_dot=(0, 200, 255), color_line=(0, 100, 255))
        
        cv2.putText(canvas, f"Frame: {frame_idx+1}/{num_frames} | FPS: {FPS:.1f}", (20, 40), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
        cv2.putText(canvas, vid_name, (20, 80), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200, 200, 200), 1)
        
        out.write(canvas)
        
    out.release()
    print(f"-> Done: {vid_name} (FPS: {FPS})")

print(f"\nDone, saved at: {OUTPUT_DIR}")

Tìm thấy 8 file .npy.
-> Done: clapping_30fps_missing.mp4 (FPS: 7.5)
-> Done: clapping_40fps_missing.mp4 (FPS: 10.0)
-> Done: shaking_30fps_missing.mp4 (FPS: 7.5)
-> Done: shaking_40fps_missing.mp4 (FPS: 10.0)
-> Done: processed_clapping_30fps_missing.mp4 (FPS: 12.5)
-> Done: processed_clapping_40fps_missing.mp4 (FPS: 12.5)
-> Done: processed_shaking_30fps_missing.mp4 (FPS: 12.5)
-> Done: processed_shaking_40fps_missing.mp4 (FPS: 12.5)

Done, saved at: data/visualize
